In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D10 — IMPI — Inquérito Mensal à Produção Industrial
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
!pip install PymuPDF

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D10"

DOCUMENT_NAME = (
    "IMPI — Inquérito Mensal à Produção Industrial"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original PDF questionnaire"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PAGE_COUNT = 4


# ------------------------------------------------------------
# Fixed Stage 1 expectations
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
}


# ------------------------------------------------------------
# Fixed Stage 1 schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
]


MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)


# ------------------------------------------------------------
# Stage 1 topic sets
# ------------------------------------------------------------

EXPECTED_INSTRUMENT_METADATA_TOPICS = [
    "Survey name",
    "Statistical system",
    "Legal basis",
    "INE registration number",
    "Validity date",
    "Electronic response URL",
    "Contact email",
    "Contact telephone"
]


EXPECTED_QUESTIONNAIRE_FIELDS = [
    "Referência dos dados",
    "NIF",
    "Número de identificação fiscal (NIF)",
    "Homepage",
    "Designação social",
    "Distrito/Ilha",
    "Município",
    "Freguesia",
    "Endereço",
    "Localidade",
    "Código postal",
    "Telefone",
    "Fax",
    "e-mail",
    "Situação na atividade",
    "Aguarda início de atividade",
    "Em atividade",
    "Atividade suspensa em",
    "Atividade cessada em",
    "Nº dias de atividade no período de referência",
    "Atividade económica principal (CAE Rev. 3)",
    "Ocorreu algum facto relevante no período de referência dos dados?",
    "Indique qual",
    "Data",
    "Observações",
    "Nome contacto",
    "Telefone",
    "Fax",
    "e-mail",
    "Função",
    "Assinatura",
    "Data"
]


EXPECTED_UAE_TEMPLATE_ELEMENTS = [
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos"
]


EXPECTED_PRODUCT_TABLE_FIELDS = [
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE"
]


EXPECTED_INSTRUCTION_TOPICS = [
    "Unidade de Atividade Económica (UAE)",
    "Questionnaire scope",
    "Unidade monetária",
    "Arredondamentos",
    "Exemplo de arredondamento",
    "Empresa",
    "Unidade de Atividade Económica (UAE)",
    "Produtos",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços"
]


# ------------------------------------------------------------
# Stage 1 expected questionnaire code assignments
# ------------------------------------------------------------

EXPECTED_CODE_BY_FIELD = {
    "Situação na atividade": "BC005",
    "Atividade suspensa em": "BC010",
    "Nº dias de atividade no período de referência": "BC007",
    "Atividade económica principal (CAE Rev. 3)": "BC001",
    "Ocorreu algum facto relevante no período de referência dos dados?": "BC015",
    "Indique qual": "BC025",
    "Data": "BC020",
    "Observações": "BC030"
}


EXPECTED_QUESTIONNAIRE_CODES = {
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
}


# ------------------------------------------------------------
# Source markers used only for input-integrity diagnostics
# ------------------------------------------------------------

SOURCE_MARKER_PATTERNS = {
    "survey_name":
        r"IMPI\s*-\s*Inquérito Mensal à Produção Industrial",

    "reference_data":
        r"Referência dos dados",

    "identification_section":
        r"Identificação da unidade estatística",

    "activity_status_section":
        r"Situação da unidade estatística",

    "observations_section":
        r"III\s+Observações",

    "responsible_person_section":
        r"Responsável pelo preenchimento",

    "uae":
        r"Unidade de Atividade Económica",

    "product_table":
        r"QUANTIDADES\s+PRODUZIDAS",

    "filling_instructions":
        r"INSTRUÇÕES DE PREENCHIMENTO",

    "explanatory_notes":
        r"NOTAS EXPLICATIVAS"
}


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D10_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_representation.json"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D10_branch_A_experiment_summary.json"
)


print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Representation:", INPUT_REPRESENTATION)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Expected reference records:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D10 PDF questionnaire."
)


uploaded = files.upload()


pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]


if len(pdf_paths) != 1:
    raise ValueError(
        "Upload exactly one PDF source document."
    )


SOURCE_PATH = pdf_paths[0]


# ------------------------------------------------------------
# File hash
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)

FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# PDF inspection
# ------------------------------------------------------------

pdf_document = fitz.open(
    SOURCE_PATH
)


PAGE_COUNT = len(
    pdf_document
)


PAGE_COUNT_VALID = (
    PAGE_COUNT == EXPECTED_PAGE_COUNT
)


page_rows = []

page_texts = []


for page_number, page in enumerate(
    pdf_document,
    start=1
):

    text = page.get_text("text") or ""

    drawings = page.get_drawings()

    images = page.get_images(
        full=True
    )

    widgets = list(
        page.widgets()
        or []
    )

    page_texts.append(
        {
            "page_number": page_number,
            "text": text
        }
    )

    page_rows.append(
        {
            "Page Number":
                page_number,

            "Native Character Count":
                len(text),

            "Native Word Count":
                len(text.split()),

            "Drawing Count":
                len(drawings),

            "Image Count":
                len(images),

            "Widget Count":
                len(widgets),

            "Width":
                float(page.rect.width),

            "Height":
                float(page.rect.height),

            "Rotation":
                int(page.rotation)
        }
    )


page_diagnostics_df = pd.DataFrame(
    page_rows
)


FULL_TEXT = "\n".join(
    item["text"]
    for item in page_texts
)


TOTAL_NATIVE_CHARACTERS = len(
    FULL_TEXT
)


TOTAL_NATIVE_WORDS = len(
    FULL_TEXT.split()
)


TEXT_EXTRACTABLE = bool(
    TOTAL_NATIVE_CHARACTERS > 100
)


OCR_REQUIRED = (
    not TEXT_EXTRACTABLE
)


# ------------------------------------------------------------
# Source marker diagnostics
# ------------------------------------------------------------

SOURCE_MARKER_STATUS = {
    name: bool(
        re.search(
            pattern,
            FULL_TEXT,
            flags=re.IGNORECASE
        )
    )

    for name, pattern
    in SOURCE_MARKER_PATTERNS.items()
}


ALL_SOURCE_MARKERS_PRESENT = all(
    SOURCE_MARKER_STATUS.values()
)


observed_questionnaire_codes = sorted(
    set(
        re.findall(
            r"\bBC\d{3}\b",
            FULL_TEXT
        )
    )
)


EXPECTED_SOURCE_CODES_PRESENT = (
    set(
        observed_questionnaire_codes
    )
    >= EXPECTED_QUESTIONNAIRE_CODES
)


# ------------------------------------------------------------
# Page-level structural diagnostics
# ------------------------------------------------------------

PAGE_FEATURES = {
    "page_1_form_layout":
        True,

    "page_1_blank_response_fields":
        True,

    "page_1_checkboxes":
        True,

    "page_1_printed_questionnaire_codes":
        True,

    "page_2_repeated_uae_blocks":
        True,

    "page_3_product_entry_grid":
        True,

    "page_3_sample_product_rows":
        True,

    "page_4_filling_instructions":
        True,

    "page_4_explanatory_notes":
        True
}


DIRECT_PDF_INGESTION_USABLE = all([
    FILE_NON_EMPTY,
    PAGE_COUNT_VALID,
    TEXT_EXTRACTABLE,
    ALL_SOURCE_MARKERS_PRESENT
])


INPUT_INTEGRITY_PASSED = (
    DIRECT_PDF_INGESTION_USABLE
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "native_word_count":
        TOTAL_NATIVE_WORDS,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "source_marker_status":
        SOURCE_MARKER_STATUS,

    "all_source_markers_present":
        ALL_SOURCE_MARKERS_PRESENT,

    "observed_questionnaire_codes":
        observed_questionnaire_codes,

    "expected_source_codes_present":
        EXPECTED_SOURCE_CODES_PRESENT,

    "page_features":
        PAGE_FEATURES,

    "direct_pdf_ingestion_usable":
        DIRECT_PDF_INGESTION_USABLE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Observed pages:",
    PAGE_COUNT
)

print(
    "Page count valid:",
    PAGE_COUNT_VALID
)

print(
    "Native characters:",
    TOTAL_NATIVE_CHARACTERS
)

print(
    "Text extractable:",
    TEXT_EXTRACTABLE
)

print(
    "OCR required:",
    OCR_REQUIRED
)

print(
    "Questionnaire codes:",
    observed_questionnaire_codes
)

print(
    "All expected source markers present:",
    ALL_SOURCE_MARKERS_PRESENT
)

print(
    "Input integrity passed:",
    INPUT_INTEGRITY_PASSED
)


display(
    page_diagnostics_df
)

if not FILE_NON_EMPTY:
    raise AssertionError(
        "D10 source PDF is empty."
    )


if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )


if not TEXT_EXTRACTABLE:
    raise AssertionError(
        "Expected D10 to contain a usable native text layer."
    )


if not ALL_SOURCE_MARKERS_PRESENT:
    raise AssertionError(
        "One or more expected source-region markers are missing."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "source_representation":
        "Structured questionnaire PDF with native text and spatial form layout",

    "complete_original_document_supplied":
        True,

    "direct_document_ingestion":
        True,

    "diagnostic_native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_layout_inspection_applied":
        True,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "page_rotation_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "form_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description": (
        "The complete original four-page D10 questionnaire PDF is "
        "submitted directly to the LLM. PyMuPDF native-text and layout "
        "inspection is used only for source-integrity diagnostics and "
        "is not supplied as an alternative model representation. "
        "No OCR, text conversion, form reconstruction, table "
        "reconstruction, structural conversion or normalisation is "
        "applied before model extraction."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

BRANCH_A_PROMPT = """You are an information extraction assistant.

Extract the predefined structural and semantic questionnaire records
represented within the defined scope of the attached original PDF:

IMPI — Inquérito Mensal à Produção Industrial.

Treat the attached original PDF as the only source of information.

The questionnaire is blank. Extract represented questionnaire
structure, labels, metadata, reusable template elements, table fields
and instructions. Do not invent or infer respondent answers.

For every included record return exactly these fields:

- Category
- Section
- Field or Concept
- Description
- Code
- Expected Value Type
- Source Location

Use exactly one of these Category values:

- Instrument metadata
- Questionnaire field
- UAE template element
- Product table field
- Instruction


1. Instrument metadata

From the header, legal notice and response-contact areas on physical
PDF page 1, extract one record for each of these predefined concepts:

- Survey name
- Statistical system
- Legal basis
- INE registration number
- Validity date
- Electronic response URL
- Contact email
- Contact telephone

Read their represented content directly from the source.

Do not create separate records from decorative branding or graphical
elements.


2. Questionnaire fields

Extract the predefined response fields represented on physical PDF
page 1.

Reference-data area:
- Referência dos dados
- NIF

Section I — Identificação da unidade estatística:
- Número de identificação fiscal (NIF)
- Homepage
- Designação social
- Distrito/Ilha
- Município
- Freguesia
- Endereço
- Localidade
- Código postal
- Telefone
- Fax
- e-mail

Section II — Situação da unidade estatística no período de referência
dos dados:
- Situação na atividade
- Aguarda início de atividade
- Em atividade
- Atividade suspensa em
- Atividade cessada em
- Nº dias de atividade no período de referência
- Atividade económica principal (CAE Rev. 3)
- Ocorreu algum facto relevante no período de referência dos dados?
- Indique qual
- Data

Section III — Observações:
- Observações

Section IV — Responsável pelo preenchimento:
- Nome contacto
- Telefone
- Fax
- e-mail
- Função
- Assinatura
- Data

Blank boxes, blank lines and empty response areas represent fields.
Do not treat them as missing respondent observations.

Where a printed questionnaire code is visibly associated with a target
field, preserve that code exactly in Code.

Use null for Code when no printed questionnaire code is represented.

Do not infer a code from a neighbouring field or from external
knowledge.


3. UAE template elements

Physical PDF page 2 contains repeated Unidade de Atividade Económica
(UAE) blocks.

Treat the repeated blocks as multiple visual instances of one reusable
template.

Extract each distinct reusable template element once:

- Código da UAE
- Designação da UAE
- Situação da UAE perante a atividade
- Observações da UAE
- Confirmar
- Produtos

Do not create duplicate records merely because the same UAE template is
displayed repeatedly.

Classify these records as:

Category = "UAE template element"
Section = "UAE information"


4. Product table fields

From the product-entry table on physical PDF page 3, extract one record
for each of these structural fields:

- NIF
- UAE
- Período de Referência
- Nº
- Produto
- Unid.
- Código
- Quantidades produzidas
- Quantidades vendidas
- Valor das vendas / prestação de serviços
- Observações empresa
- Observações INE

Preserve the table-column structure represented by the source.

The rows labelled Produto a, Produto b, Produto c and Produto d are
sample/template examples. Do not extract them as respondent
observations or additional product records.

Do not extract their sample product codes or sample unit letters as
separate observations.


5. Instructions and explanatory definitions

Extract the principal predefined instructional and explanatory concepts
represented on physical PDF pages 2 and 4.

From the UAE information on page 2:
- Unidade de Atividade Económica (UAE)

From the filling instructions on page 4:
- Questionnaire scope
- Unidade monetária
- Arredondamentos
- Exemplo de arredondamento

From the explanatory notes on page 4:
- Empresa
- Unidade de Atividade Económica (UAE)
- Produtos
- Quantidades produzidas
- Quantidades vendidas
- Valor das vendas / prestação de serviços

For each instruction or definition, provide a concise source-grounded
Description preserving the substantive represented rule or definition.

Do not split supporting sentences into additional records outside this
predefined scope.


Field rules:

Category:
- Use exactly one of the five Category labels defined above.

Section:
- Identify the source section or structural region containing the
  represented element.
- Use concise stable section labels.

Field or Concept:
- Use the predefined field or concept label corresponding to the item
  being extracted.
- Preserve Portuguese source wording where the item is a visible source
  label.

Description:
- Provide a concise source-grounded description of the represented
  field, metadata item, template element, table field or instruction.
- Do not add external interpretation.
- For instructions and definitions, retain the substantive meaning
  explicitly represented by the source.

Code:
- Preserve a printed questionnaire code only when visibly associated
  with the represented target element.
- Use null when no printed code applies.
- Do not infer or transfer codes between neighbouring elements.

Expected Value Type:
- Assign a concise structural expected-value type based only on the
  visible questionnaire element or semantic role of the target item.
- Use consistent labels such as:
  text
  identifier
  numeric identifier
  date
  URL
  email
  telephone number
  reference period
  postal code
  category
  checkbox
  integer
  CAE code
  yes or no
  free text
  signature
  button or action
  month or period
  unit
  product code
  numeric quantity
  monetary value
  instruction
  definition
  example
- Do not infer actual respondent values.

Source Location:
- Use physical PDF page references grounded in the supplied source.
- Include a concise source region or section.
- Examples include:
  "PDF page 1 — Header"
  "PDF page 1 — Reference data"
  "PDF page 1 — Section I"
  "PDF page 1 — Section II"
  "PDF page 1 — Section III"
  "PDF page 1 — Section IV"
  "PDF page 2 — UAE information"
  "PDF page 3 — Product production table"
  "PDF page 4 — Filling instructions"
  "PDF page 4 — Explanatory notes"


Additional extraction rules:

- Use only information explicitly represented in the original PDF.
- Preserve Portuguese wording, labels and printed codes where relevant.
- Do not invent respondent answers from blank response fields.
- Do not treat blank response areas as null observations.
- Do not duplicate repeated UAE template elements.
- Do not treat sample product rows as respondent observations.
- Do not calculate, infer, derive, repair or introduce information.
- Do not use external knowledge.
- Do not follow external links.
- Do not reconstruct hidden form data.
- Do not create records outside the predefined extraction scope.
- Verify that every item within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D10",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Section": null,
      "Field or Concept": null,
      "Description": null,
      "Code": null,
      "Expected Value Type": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open new independent ChatGPT conversation.

Upload:

1. the complete original D10 PDF;
2. `D10_branch_A_prompt.txt`.

Submission of the prompt once.

Save the complete untouched model response as:

`D10_branch_A_raw_response.txt`

In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D10_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".txt")
]


if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():
    raise ValueError(
        "The raw response is empty."
    )


# ------------------------------------------------------------
# Preserve raw response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Non-crashing JSON parse
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(error)


# ------------------------------------------------------------
# Top-level standardized wrapper
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Parsed artifact only when records are evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True

    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Exact schema and field order
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Field names or field order differ",

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field
                            for field in EXPECTED_FIELDS
                            if field not in record
                        ],

                    "extra_fields":
                        [
                            field
                            for field in observed_fields
                            if field not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue["record_index"]
        for issue in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types + mandatory completeness
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(value).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(field)

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue["record_index"]
        for issue in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Record count + categories
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# D. Full-record duplicates
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )


    duplicate_records = [
        list(key)
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# E. Generic helpers
# ------------------------------------------------------------

def normalized_text(value):

    if value is None:
        return None

    return str(value).strip().casefold()


def matching_records(
    category=None,
    section=None,
    field_or_concept=None
):

    if not records_evaluable:
        return []


    matches = []


    for record in extracted_records:

        if not isinstance(record, dict):
            continue


        if (
            category is not None
            and record.get("Category")
            != category
        ):
            continue


        if (
            section is not None
            and record.get("Section")
            != section
        ):
            continue


        if (
            field_or_concept is not None
            and record.get("Field or Concept")
            != field_or_concept
        ):
            continue


        matches.append(record)


    return matches


def unique_record(
    category,
    section,
    field_or_concept
):

    matches = matching_records(
        category=category,
        section=section,
        field_or_concept=field_or_concept
    )


    if len(matches) == 1:
        return matches[0]


    return None


# ------------------------------------------------------------
# F. Expected-value-type distribution
# ------------------------------------------------------------

if records_evaluable:

    expected_value_type_counts = dict(
        Counter(
            record.get(
                "Expected Value Type"
            )
            for record in extracted_records
            if isinstance(record, dict)
        )
    )


else:

    expected_value_type_counts = None


# ------------------------------------------------------------
# G. Instrument metadata topics
# ------------------------------------------------------------

if records_evaluable:

    instrument_metadata_status = {
        topic: (
            len(
                matching_records(
                    category="Instrument metadata",
                    field_or_concept=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_INSTRUMENT_METADATA_TOPICS
    }


    instrument_metadata_complete = all(
        instrument_metadata_status.values()
    )


else:

    instrument_metadata_status = None

    instrument_metadata_complete = None


# ------------------------------------------------------------
# H. Reference-period field
# ------------------------------------------------------------

if records_evaluable:

    reference_period_record = unique_record(
        "Questionnaire field",
        "Reference data",
        "Referência dos dados"
    )


    reference_period_field_present = (
        reference_period_record is not None
    )


else:

    reference_period_field_present = None


# ------------------------------------------------------------
# I. UAE reusable-template diagnostics
# ------------------------------------------------------------

if records_evaluable:

    uae_template_element_status = {
        element: (
            len(
                matching_records(
                    category="UAE template element",
                    section="UAE information",
                    field_or_concept=element
                )
            )
            == 1
        )
        for element
        in EXPECTED_UAE_TEMPLATE_ELEMENTS
    }


    uae_template_complete = all(
        uae_template_element_status.values()
    )


    observed_uae_template_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category")
            == "UAE template element"
        )
    )


    uae_template_not_duplicated = (
        observed_uae_template_count
        == len(
            EXPECTED_UAE_TEMPLATE_ELEMENTS
        )
    )


else:

    uae_template_element_status = None

    uae_template_complete = None

    observed_uae_template_count = None

    uae_template_not_duplicated = None


# ------------------------------------------------------------
# J. Product table structural fields
# ------------------------------------------------------------

if records_evaluable:

    product_table_field_status = {
        field: (
            len(
                matching_records(
                    category="Product table field",
                    section="Product production table",
                    field_or_concept=field
                )
            )
            == 1
        )
        for field
        in EXPECTED_PRODUCT_TABLE_FIELDS
    }


    product_table_scope_complete = all(
        product_table_field_status.values()
    )


else:

    product_table_field_status = None

    product_table_scope_complete = None


# ------------------------------------------------------------
# K. Sample-product exclusion
# ------------------------------------------------------------

SAMPLE_PRODUCT_TOKENS = [
    "Produto a",
    "Produto b",
    "Produto c",
    "Produto d",
    "111111111111"
]


if records_evaluable:

    sample_product_issues = []


    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(record, dict):
            continue


        for field in EXPECTED_FIELDS:

            value = record.get(field)

            if not isinstance(value, str):
                continue


            for sample_token in SAMPLE_PRODUCT_TOKENS:

                if (
                    sample_token.casefold()
                    in value.casefold()
                ):

                    sample_product_issues.append(
                        {
                            "record_index":
                                record_index,

                            "field":
                                field,

                            "sample_token":
                                sample_token,

                            "observed_value":
                                value
                        }
                    )


    sample_product_issue_count = len(
        sample_product_issues
    )


    sample_product_rows_excluded = (
        sample_product_issue_count == 0
    )


else:

    sample_product_issues = None

    sample_product_issue_count = None

    sample_product_rows_excluded = None


# ------------------------------------------------------------
# L. Questionnaire code preservation
# ------------------------------------------------------------

if records_evaluable:

    observed_non_null_codes = sorted(
        {
            record.get("Code")
            for record in extracted_records
            if (
                isinstance(record, dict)
                and isinstance(
                    record.get("Code"),
                    str
                )
                and record.get("Code").strip()
            )
        }
    )


    observed_code_set = set(
        observed_non_null_codes
    )


    questionnaire_code_set_valid = (
        observed_code_set
        == EXPECTED_QUESTIONNAIRE_CODES
    )


    questionnaire_code_assignment_status = {}


    for field_label, expected_code in (
        EXPECTED_CODE_BY_FIELD.items()
    ):

        matching_field_records = [
            record
            for record in extracted_records
            if (
                isinstance(record, dict)
                and record.get("Category")
                == "Questionnaire field"
                and record.get("Field or Concept")
                == field_label
                and record.get("Code")
                == expected_code
            )
        ]


        questionnaire_code_assignment_status[
            f"{field_label} -> {expected_code}"
        ] = (
            len(matching_field_records)
            == 1
        )


    questionnaire_code_assignments_valid = all(
        questionnaire_code_assignment_status.values()
    )


else:

    observed_non_null_codes = None

    questionnaire_code_set_valid = None

    questionnaire_code_assignment_status = None

    questionnaire_code_assignments_valid = None


# ------------------------------------------------------------
# M. Instruction category coverage
# ------------------------------------------------------------

EXPECTED_INSTRUCTION_KEYS = [
    (
        "UAE information",
        "Unidade de Atividade Económica (UAE)"
    ),
    (
        "Filling instructions",
        "Questionnaire scope"
    ),
    (
        "Filling instructions",
        "Unidade monetária"
    ),
    (
        "Filling instructions",
        "Arredondamentos"
    ),
    (
        "Filling instructions",
        "Exemplo de arredondamento"
    ),
    (
        "Explanatory notes",
        "Empresa"
    ),
    (
        "Explanatory notes",
        "Unidade de Atividade Económica (UAE)"
    ),
    (
        "Explanatory notes",
        "Produtos"
    ),
    (
        "Explanatory notes",
        "Quantidades produzidas"
    ),
    (
        "Explanatory notes",
        "Quantidades vendidas"
    ),
    (
        "Explanatory notes",
        "Valor das vendas / prestação de serviços"
    )
]


if records_evaluable:

    instruction_status = {
        f"{section} | {concept}": (
            len(
                matching_records(
                    category="Instruction",
                    section=section,
                    field_or_concept=concept
                )
            )
            == 1
        )
        for section, concept
        in EXPECTED_INSTRUCTION_KEYS
    }


    instruction_scope_complete = all(
        instruction_status.values()
    )


else:

    instruction_status = None

    instruction_scope_complete = None


# ------------------------------------------------------------
# N. Content diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "instrument_metadata_status":
        instrument_metadata_status,

    "instrument_metadata_complete":
        instrument_metadata_complete,

    "reference_period_field_present":
        reference_period_field_present,

    "uae_template_element_status":
        uae_template_element_status,

    "observed_uae_template_count":
        observed_uae_template_count,

    "uae_template_complete":
        uae_template_complete,

    "uae_template_not_duplicated":
        uae_template_not_duplicated,

    "product_table_field_status":
        product_table_field_status,

    "product_table_scope_complete":
        product_table_scope_complete,

    "sample_product_issue_count":
        sample_product_issue_count,

    "sample_product_rows_excluded":
        sample_product_rows_excluded,

    "observed_non_null_codes":
        observed_non_null_codes,

    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,

    "questionnaire_code_assignment_status":
        questionnaire_code_assignment_status,

    "questionnaire_code_assignments_valid":
        questionnaire_code_assignments_valid,

    "instruction_status":
        instruction_status,

    "instruction_scope_complete":
        instruction_scope_complete,

    "expected_value_type_counts":
        expected_value_type_counts
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches reference:",
    record_count_valid
)

print(
    "Category counts match reference:",
    category_counts_valid
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

print(
    "Duplicate records:",
    duplicate_record_count
)

print(
    "Referência dos dados present:",
    reference_period_field_present
)

print(
    "UAE template complete:",
    uae_template_complete
)

print(
    "UAE template not duplicated:",
    uae_template_not_duplicated
)

print(
    "Product-table scope complete:",
    product_table_scope_complete
)

print(
    "Sample product rows excluded:",
    sample_product_rows_excluded
)

print(
    "Questionnaire code set valid:",
    questionnaire_code_set_valid
)

print(
    "Questionnaire code assignments valid:",
    questionnaire_code_assignments_valid
)

print(
    "Instruction scope complete:",
    instruction_scope_complete
)


print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts
    is not None
    else None
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(valid_json),

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


# ------------------------------------------------------------
# Structure-check artifact
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(field_type_issues)
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            PAGE_COUNT_VALID,

        "text_extractable":
            TEXT_EXTRACTABLE,

        "ocr_required":
            OCR_REQUIRED,

        "contains_form_layout":
            True,

        "contains_blank_response_fields":
            True,

        "contains_checkboxes":
            True,

        "contains_printed_questionnaire_codes":
            True,

        "contains_repeated_uae_blocks":
            True,

        "contains_product_entry_grid":
            True,

        "contains_sample_product_rows":
            True,

        "contains_explanatory_instructions":
            True
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_layout_inspection_applied":
        True,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "page_rotation_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "form_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_pdf_supplied":
        True,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "reference_schema_version":
        "v1",

    "reference_task_version":
        "v1",

    "notes": (
        "Branch A submits the complete original four-page D10 "
        "questionnaire PDF directly to the model. PyMuPDF native-text "
        "and layout inspection is used only for source-integrity "
        "diagnostics and is not supplied as an alternative model "
        "representation. No OCR, PDF-to-text conversion, page "
        "extraction, cropping, form reconstruction, table "
        "reconstruction, structural conversion, normalisation or "
        "manual correction is applied before extraction. Stage 1 "
        "reference values, expected record count, expected category "
        "distribution and expected code assignments are not supplied "
        "to the model. Content-level validation is performed separately in "
        "Validation A — D10."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "instrument_metadata_complete":
        instrument_metadata_complete,

    "reference_period_field_present":
        reference_period_field_present,

    "uae_template_complete":
        uae_template_complete,

    "uae_template_not_duplicated":
        uae_template_not_duplicated,

    "product_table_scope_complete":
        product_table_scope_complete,

    "sample_product_rows_excluded":
        sample_product_rows_excluded,

    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,

    "questionnaire_code_assignments_valid":
        questionnaire_code_assignments_valid,

    "instruction_scope_complete":
        instruction_scope_complete,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, representation "
        "characterisation, D10 Branch A direct-PDF execution "
        "preservation, technical/schema checks and document-specific "
        "content diagnostics only. Formal agreement with the fixed "
        "Stage 1 reference dataset is evaluated separately in "
        "Validation A — D10."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final display
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nStructurally evaluable:",
    structurally_evaluable
)


print(
    "\n" + "=" * 60
)

print(
    "D10 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed        :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Text extractable              :",
    TEXT_EXTRACTABLE
)

print(
    "OCR required                  :",
    OCR_REQUIRED
)

print(
    "Raw response preserved        :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                    :",
    valid_json
)

print(
    "Records evaluable             :",
    records_evaluable
)

print(
    "Expected records              :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records              :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches          :",
    record_count_valid
)

print(
    "Category counts match         :",
    category_counts_valid
)

print(
    "Record schema valid           :",
    record_schema_valid
)

print(
    "Field types valid             :",
    field_types_valid
)

print(
    "Structurally evaluable       :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step                     : Validation A — D10"
)


# ------------------------------------------------------------
# Output existence checks
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):
    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name
    for path in required_output_paths
    if not path.exists()
]


if missing_output_files:
    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D10 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )